# 优势函数们DP vs MC vs TD VS GAE

动态规划、蒙特卡洛与时序差分 还有GAE

# DP、MC、TD 的共同核心是贝尔曼思想：当前价值可以由即时奖励和未来价值来刻画。它们的差别在于未来价值是由模型精确计算、由完整经历给出，还是由下一状态估计近似


理解这三种求解价值函数的核心算法，关键在于搞懂两个维度：
1. **需不需要环境的绝对上帝视角？**（是否已知环境模型/是否需要采样）
2. **更新时是“走到底再算总账”，还是“走一步看一步”？**（是否自举 Bootstrapping）

---

## 1. 动态规划 (Dynamic Programming, DP)

* **核心特点**：**已知模型、全量更新、走一步看一步。**
* **前提条件**：必须完全掌握环境的运转规则（即状态转移概率 $P$ 和奖励函数 $R$），属于“白盒”环境。
* **计算方式**：不需要与环境实际交互，而是在脑海中遍历所有可能的分支计算期望。
* **核心公式**（DP 策略评估）：
  $$V_{k+1}(s) \leftarrow \sum_a \pi(a \mid s) \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V_k(s') \right]$$
  * **解析**：利用已知的世界模型 $P(s' \mid s, a)$ 进行完美推演，并用下一步的估计值 $V_k(s')$ 更新当前的 $V_{k+1}(s)$，这称为**自举（Bootstrapping）**。

## 2. 蒙特卡洛 (Monte Carlo, MC)

* **核心特点**：**未知模型、真实采样、走到底算总账。**
* **前提条件**：无需知道环境模型（黑盒环境），通过在环境中不断试错、采样完整轨迹来积累经验。
* **计算方式**：必须把整个回合（Episode）走完，拿到最终的真实总回报（Return, $G_t$），再回溯更新沿途的状态价值。
* **核心公式**（MC 更新）：
  $$V(s) \leftarrow V(s) + \alpha [G_t - V(s)]$$
  * **解析**：$G_t$ 是整条轨迹跑完后的真实完整回报。由于**没有自举**，结果无偏（Unbiased），但因为每局轨迹差异大，方差（Variance）较高。

## 3. 时序差分 (Temporal Difference, TD)

* **核心特点**：**未知模型、真实采样、走一步看一步。**
* **算法地位**：结合了 DP 和 MC 的优点，是现代强化学习（如 Q-Learning）的基石。
* **计算方式**：不需要环境模型（实机采样），且**不需要等回合结束**。走一步拿到即时奖励 $r$，结合对下一步的“旧估计” $V(s')$，立刻更新当前状态价值。
* **核心公式**：
  * **TD(0) 更新**：
    $$V(s) \leftarrow V(s) + \alpha [r + \gamma V(s') - V(s)]$$
  * **TD Error**（TD 误差）：
    $$\delta = r + \gamma V(s') - V(s)$$
  * **解析**：用 $r + \gamma V(s')$（即 TD 目标）代替 MC 中的真实回报 $G_t$。深度强化学习的核心通常就是最小化这个 TD 误差 $\delta$。

---

## 💡 核心总结对比

| 算法 | 是否需要环境模型？(Model-free?) | 更新深度 (是否自举 Bootstrapping?) | 优缺点对比 |
| :--- | :--- | :--- | :--- |
| **DP** | ❌ **需要** (Model-based) | ✅ **走一步看一步** (自举) | 理论完美，但现实中极难获知精准模型，且存在维度灾难。 |
| **MC** | ✅ **不需要** (无模型采样) | ❌ **走到结局才更新** (无自举) | 无偏，但方差大（单次采样随机性高），且必须等回合结束才能更新。 |
| **TD** | ✅ **不需要** (无模型采样) | ✅ **走一步看一步** (自举) | 支持在线实时更新，方差较小。结合了 DP 和 MC 的优点，是 RL 主流。 |

值得注意：这里虽然R(s,a) 维护的是一个奖励表格，不同状态S不同动作a，带来的奖励都不同，但是DP是都知道的是有这一张表格的，但是TD是不需要维护这么一张表格的
所以DP是需要环境模型，但是TD不需要的
---

## 🚗 形象的比喻：预测通勤时间

* **动态规划 (DP)**：你有完美的城市交通流控图表，知道每个路口绿灯和拥堵的精确概率，在家拿笔算出各条路线的数学期望。
* **蒙特卡洛 (MC)**：你不看图表，亲自开车跑，到了公司看手表记下总时间（算总账）。跑100次取平均值。
* **时序差分 (TD)**：你开车上路，刚过一个拥堵路口花了 10 分钟。你看了看剩下的路程，心想“按过去经验，剩下的路大概还要 20 分钟”，于是你立刻在车上调整了今天的总时间预测（10+20=30分钟）。

# 三者关键区别（非常重要）

都要维系一个 v 价值表，最开始因为不知道整个价值表，所以说他会使用一个估计价值来替代未知的真实的价值，而三者的区别就在于如何算这个估计价值 target

DP下一步使用估计的，MC是一口气全部用真实的算出来， TD是下一步用真实的 + 下下步后面的全部使用估计

三种经典方法的核心差异，就在于如何构造这个 **target**。

### 1. DP（动态规划）
DP 假设已知完整的环境模型。它利用贝尔曼期望方程，把策略下所有动作分支和下一状态分支全部展开，直接计算期望。因为模型已经给出了所有分支的概率和奖励，不需要实际进入环境采样，遍历一遍状态表即可完成一轮更新。

### 2. MC（蒙特卡洛）
MC 不假设知道模型。它等到一次完整的 episode 结束后，把实际发生的总回报 $G_t$ 直接当作 target。$G_t$ 是这条轨迹上从该状态出发的真实回报样本，不是模型算出的平均值。它的更新只能在 episode 结束后进行。

### 3. TD（时序差分）
TD 同样不知道模型，但它不等 episode 结束。每走一步，它就把当前观察到的即时奖励 $R_{t+1}$ 和下一状态的当前估计值 $V(S_{t+1})$ 组合成 target，即 $R_{t+1} + \gamma V(S_{t+1})$。

TD 能这样做的原因是回报具有递归结构：

$$
G_t = R_{t+1} + \gamma G_{t+1}
$$

$G_{t+1}$ 是从下一状态开始的完整回报。在 $t+1$ 时刻，$G_{t+1}$ 尚未发生，TD 就用表里对下一状态的当前估计 $V(S_{t+1})$ 代替它：

$$
\text{target}_{\text{TD}} = R_{t+1} + \gamma V(S_{t+1})
$$

这一步叫**自举（bootstrapping）**。它让 TD 能一步一更新，也会带来后面要讨论的偏差。

---

### 统一更新框架

三种方法虽然 target 不同，但更新都可以归入同一个框架：把当前估计向 target 移动。

$$
V(s) \leftarrow V(s) + \alpha \big[ \text{target} - V(s) \big]
$$

- **$V(s)$**：表里的旧数字。
- **target**：这次算出来的新估计。
- **$\alpha$**：控制步幅。

$\text{target} - V(s)$ 表示“这次认为旧表错了多少”；乘上 $\alpha$，就是这次实际改多少。$\alpha=1$ 时直接覆盖旧值，$\alpha$ 较小时只往目标方向挪一小步。

> **总结**：本节的重点不是分别学三种算法，而是比较三种构造 target 的方式。三种方法都在改同一张价值表，区别只在 target 的来源。

直接阅读这个帖子 ： https://walkinglabs.github.io/hands-on-modern-rl/chapter03_mdp/dp-mc-td

TD误差 = [ 一步贝尔曼目标 ($r + \gamma V(s')$) ] - [ 价值估计 ($V(s)$) ]

而DP是知道环境模型，知道在每个环境下做什么动作可以切换到下一个环境，这个情况下就可以枚举，使用DP

不知道环境模型使用MC 和 TD， MC是全程随机动作，然后拿到奖励，再用奖励总和来更新当前状态的 V(s)。多次尝试以后拿到多个VS来跟来不断逼近真实的VS。

Td 是用当前的这一步真实奖励和下一步的一个预估 V_s 来更新当前的 V_s，这样就不用从头到尾更新

In [1]:
import random

STATES = ["S", "M", "G"]
GAMMA = 1.0


def step(state, action):
    # 环境本身是确定的：给定状态和动作，下一状态、奖励都固定。
    # 随机性只来自后面的策略 sample_action()。
    if state == "S":
        return ("M", -1) if action == "right" else ("S", -2)
    if state == "M":
        return ("G", -1) if action == "right" else ("S", -2)
    return "G", 0


def sample_action():
    # 固定策略 pi：80% 向右，20% 向左。
    return "right" if random.random() < 0.8 else "left"


def dp_policy_evaluation(n_iter=1_000):
    V = {s: 0.0 for s in STATES}
    for _ in range(n_iter):
        # DP 知道模型，因此可以直接枚举两个动作分支。
        # 这里用 old 做同步更新：本轮读旧表，写出新表。
        old = V.copy()
        V["S"] = 0.8 * (-1 + GAMMA * old["M"]) + 0.2 * (-2 + GAMMA * old["S"])
        V["M"] = 0.8 * (-1 + GAMMA * old["G"]) + 0.2 * (-2 + GAMMA * old["S"])
        V["G"] = 0.0
    return V


def generate_episode():
    # MC 和 TD 都不知道模型，只能让智能体真的走一局。
    episode = []
    state = "S"
    while state != "G":
        action = sample_action()
        next_state, reward = step(state, action)
        episode.append((state, reward, next_state))
        state = next_state
    return episode


def mc_every_visit(n_episodes=1_000_000, seed=0):
    random.seed(seed)
    V = {s: 0.0 for s in STATES}
    N = {s: 0 for s in STATES}
    for _ in range(n_episodes):
        episode = generate_episode()
        G = 0.0
        # MC 等整局结束后，从后往前累加完整回报 G_t。
        for state, reward, _ in reversed(episode):
            G = reward + GAMMA * G
            N[state] += 1
            # 每次访问都更新；1/N 是样本平均的增量写法。
            V[state] += (G - V[state]) / N[state]
    return V


def td_zero(n_episodes=1_000_000, seed=0):
    random.seed(seed)
    V = {s: 0.0 for s in STATES}
    N = {s: 0 for s in STATES}
    for _ in range(n_episodes):
        state = "S"
        while state != "G":
            action = sample_action()
            next_state, reward = step(state, action)
            N[state] += 1
            alpha = 1.0 / N[state]
            # TD 不等整局结束：一步奖励 + 下一状态当前估计。
            target = reward + GAMMA * V[next_state]
            V[state] += alpha * (target - V[state])
            state = next_state
    return V


def show(name, values):
    print(f"{name}: S={values['S']:.6f}, M={values['M']:.6f}, G={values['G']:.6f}")


def summarize(name, runs):
    mean_s = sum(v["S"] for v in runs) / len(runs)
    mean_m = sum(v["M"] for v in runs) / len(runs)
    min_s, max_s = min(v["S"] for v in runs), max(v["S"] for v in runs)
    min_m, max_m = min(v["M"] for v in runs), max(v["M"] for v in runs)
    print(
        f"{name}: mean S={mean_s:.6f} [{min_s:.6f}, {max_s:.6f}], "
        f"mean M={mean_m:.6f} [{min_m:.6f}, {max_m:.6f}]"
    )


print("single run")
show("DP", dp_policy_evaluation())
show("MC", mc_every_visit(seed=0))
show("TD", td_zero(seed=0))

print("\n5-run summary")
seeds = range(5)
summarize("MC", [mc_every_visit(seed=s) for s in seeds])
summarize("TD", [td_zero(seed=s) for s in seeds])

single run
DP: S=-3.375000, M=-1.875000, G=0.000000
MC: S=-3.373813, M=-1.874359, G=0.000000
TD: S=-3.374871, M=-1.874966, G=0.000000

5-run summary
MC: mean S=-3.374261 [-3.376874, -3.372061], mean M=-1.874122 [-1.874833, -1.872401]
TD: mean S=-3.375231 [-3.380956, -3.366307], mean M=-1.874380 [-1.876858, -1.870551]


# GAE

GAE就是TD 和 MC的结合版本， GAE中引入了一个lambda到优势函数当中，将MC 和 TD结合起来 

**1. GAE 的统一表达式**

广义优势估计（Generalized Advantage Estimation，GAE）可以写成：

$$
\hat A_t^{\mathrm{GAE}(\gamma,\lambda)}
=\sum_{k=0}^{\infty}(\gamma\lambda)^k\delta_{t+k}.
$$

其中：

- \(\gamma\) 是奖励折扣因子；
- \(\lambda\) 控制远期 TD 误差的衰减速度；
- \(\delta_t\) 是一步 TD 误差。

一步 TD 误差定义为：

$$
\delta_t=r_t+\gamma V(s_{t+1})-V(s_t).
$$

---

**2. 不同 \(\lambda\) 的含义**

**\(\lambda=0\)：退化为一步 TD**

当 \(\lambda=0\) 时：

$$
\hat A_t
=\sum_{k=0}^{\infty}(\gamma\cdot 0)^k\delta_{t+k}
=\delta_t.
$$

因此：

$$
\boxed{\hat A_t=\delta_t}
$$

此时只使用当前时刻的一步 TD 误差，不考虑未来的 TD 误差。

**\(\lambda=1\)：退化为 MC 优势**

当 \(\lambda=1\) 时：

$$
\hat A_t
=\sum_{k=0}^{\infty}\gamma^k\delta_{t+k}.
$$

在终止状态满足 \(V(s_T)=0\) 的条件下：

$$
\sum_{k=0}^{\infty}\gamma^k\delta_{t+k}
=G_t-V(s_t).
$$

因此：

$$
\boxed{\hat A_t=G_t-V(s_t)}
$$

这就是蒙特卡洛优势估计。

> 注意：并不是 MC 回报 \(G_t\) 本身等于 TD 误差之和，而是 MC 优势 \(G_t-V(s_t)\) 等于未来 TD 误差的折扣和。

**\(0<\lambda<1\)：在 TD 与 MC 之间折中**

当 \(0<\lambda<1\) 时，未来的 \(\delta_{t+k}\) 按照

$$
(\gamma\lambda)^k
$$

指数衰减：

$$
\hat A_t
=\delta_t
+\gamma\lambda\delta_{t+1}
+(\gamma\lambda)^2\delta_{t+2}
+\cdots.
$$

因此：

- \(\lambda\) 越小，越接近一步 TD，方差通常较低，但偏差可能较大；
- \(\lambda\) 越大，越接近 MC，偏差通常较小，但方差可能较大。

---

**3. 为什么 MC 优势等于 TD 误差的折扣和**

一步 TD 误差为：

$$
\delta_t=r_t+\gamma V(s_{t+1})-V(s_t).
$$

将未来的 TD 误差按 \(\gamma\) 折扣展开：

$$
\begin{aligned}
\delta_t
&=r_t+\gamma V(s_{t+1})-V(s_t),\\
\gamma\delta_{t+1}
&=\gamma r_{t+1}+\gamma^2V(s_{t+2})
-\gamma V(s_{t+1}),\\
\gamma^2\delta_{t+2}
&=\gamma^2r_{t+2}+\gamma^3V(s_{t+3})
-\gamma^2V(s_{t+2}),\\
&\ \vdots
\end{aligned}
$$

将这些式子相加：

$$
\begin{aligned}
&\delta_t+\gamma\delta_{t+1}+\gamma^2\delta_{t+2}+\cdots\\
={}&r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\cdots\\
&-V(s_t)\\
&+\cancel{\gamma V(s_{t+1})}
-\cancel{\gamma V(s_{t+1})}\\
&+\cancel{\gamma^2V(s_{t+2})}
-\cancel{\gamma^2V(s_{t+2})}\\
&+\cdots.
\end{aligned}
$$

中间状态的价值项会两两抵消，这种结构称为**望远镜求和（telescoping sum）**。

对于一条在时刻 \(T\) 结束的轨迹，有：

$$
\sum_{k=0}^{T-t-1}\gamma^k\delta_{t+k}
=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}
-V(s_t)
+\gamma^{T-t}V(s_T).
$$

蒙特卡洛回报定义为：

$$
G_t=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.
$$

因此：

$$
\sum_{k=0}^{T-t-1}\gamma^k\delta_{t+k}
=G_t-V(s_t)+\gamma^{T-t}V(s_T).
$$

如果 \(s_T\) 是终止状态，通常规定：

$$
V(s_T)=0.
$$

最终得到：

$$
\boxed{
\sum_{k=0}^{T-t-1}\gamma^k\delta_{t+k}
=G_t-V(s_t)
}
$$

---

**4. 从权重角度理解 TD、MC 和 GAE**

将优势估计统一写成：

$$
\hat A_t=\sum_{k=0}^{\infty}w_k\delta_{t+k}.
$$

三种方法对应的权重分别为：

| 方法 | 权重 \(w_k\) | 含义 |
|---|---:|---|
| 一步 TD | \(w_0=1,\;w_{k>0}=0\) | 只使用当前 TD 误差 |
| MC 优势 | \(w_k=\gamma^k\) | 使用所有未来 TD 误差 |
| GAE | \(w_k=(\gamma\lambda)^k\) | 对远期 TD 误差进行额外衰减 |

因此，GAE 可以理解为：在 MC 原有的折扣权重 \(\gamma^k\) 上，再乘一个由 \(\lambda\) 控制的衰减因子 \(\lambda^k\)：

$$
\gamma^k\lambda^k=(\gamma\lambda)^k.
$$

---

**5. 核心结论**

$$
\boxed{
\hat A_t^{\mathrm{GAE}}
=\sum_{k=0}^{\infty}(\gamma\lambda)^k\delta_{t+k}
}
$$

- \(\lambda=0\)：只保留当前 TD 误差，退化为一步 TD；
- \(\lambda=1\)：累加所有未来 TD 误差，退化为 MC 优势；
- \(0<\lambda<1\)：在 TD 的低方差与 MC 的低偏差之间折中；
- MC 优势并非与 TD 完全独立的另一种信号，而是未来一步 TD 误差的折扣累加。


In [ ]:
# ==========================================
# 手动实现 GAE 计算
# ==========================================
import numpy as np

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    计算 GAE（广义优势估计）

    参数:
        rewards: 每一步的即时奖励列表
        values: Critic 对每个状态的估计值 V(s)
        dones: 每一步是否结束
        gamma: 折扣因子
        lam: GAE 的 λ 参数

    返回:
        advantages: 每一步的优势估计
        returns: 每一步的目标回报（用于训练 Critic）
    """
    
    advantages = []
    gae = 0  # 累积的 GAE 值

    # 从后往前计算（因为 Â_t 依赖后续的 δ）
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0  # 最后一步的下一个状态价值为 0
        else:
            next_value = values[t + 1]

        # TD Error: δ_t = r_t + γ * V(s_{t+1}) - V(s_t)
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]

        # GAE 累积：Â_t = δ_t + γλ * δ_{t+1} + (γλ)² * δ_{t+2} + ...
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)

    # 目标回报 = 优势 + 价值估计
    advantages = np.array(advantages)
    returns = advantages + np.array(values[:len(rewards)])

    return advantages, returns

# 一个 5 步的 episode
rewards = [0.0, 0.0, 0.0, 0.0, 1.0]  # 只有最后一步有奖励
values  = [0.1, 0.2, 0.3, 0.5, 0.8]  # Critic 的估计值
dones   = [0,   0,   0,   0,   1  ]  # 只有最后一步结束

advantages, returns = compute_gae(rewards, values, dones)
print("优势估计:", advantages)
print("目标回报:", returns)